In [ ]:
# uv add qdrant-client==1.16.1
# 部署
# docker run -d -p 6333:6333 qdrant/qdrant

In [17]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams,Distance,PointStruct,Filter,FieldCondition, MatchValue
import numpy as np

In [2]:
# 创建客户端
client = QdrantClient(host="localhost", port=6333)

In [5]:
# 准备元数据
items = [
    {"type": "phone", "id": "用户A", "number": 13800001234},
    {"type": "phone", "id": "用户B", "number": 13800005678},
    {"type": "order", "id": "订单1001", "number": 203011010001},
    {"type": "order", "id": "订单1002", "number": 203011010123},
    {"type": "order", "id": "订单2001", "number": 203012150045},
    {"type": "phone", "id": "用户C", "number": 13912345678},
    {"type": "phone", "id": "用户D", "number": 13798765432},
    {"type": "order", "id": "订单3001", "number": 205001020333},
    {"type": "order", "id": "订单3002", "number": 205001020777},
    {"type": "phone", "id": "用户E", "number": 13622223333},
]
numbers = np.array([row['number'] for row in items],dtype ='float32')
numbers

array([1.3800002e+10, 1.3800006e+10, 2.0301101e+11, 2.0301101e+11,
       2.0301215e+11, 1.3912346e+10, 1.3798766e+10, 2.0500102e+11,
       2.0500102e+11, 1.3622223e+10], dtype=float32)

In [ ]:
# 向量的维度
dimension = 1
vectors = numbers.reshape(-1,1)  # shape(N,1) 
vectors

array([[1.3800002e+10],
       [1.3800006e+10],
       [2.0301101e+11],
       [2.0301101e+11],
       [2.0301215e+11],
       [1.3912346e+10],
       [1.3798766e+10],
       [2.0500102e+11],
       [2.0500102e+11],
       [1.3622223e+10]], dtype=float32)

In [9]:
# 集合的名称
collection_name = "numbers_collection"
# 创建集合
client.create_collection(
    collection_name,
    vectors_config=VectorParams(size=dimension, distance=Distance.EUCLID)
    )

True

In [12]:
# 准备数据
points = []
for idx ,item in enumerate(items,start =1):
    vec = [float(item['number'])]
    points.append(
        PointStruct(
            id = idx,
            vector = vec,
            payload = {
                "type":item['type'],
                "id":item['id'],
                "number":item['number']
            }
        )
    )
client.upsert(collection_name,points = points)
print(f"已向索引添加 {len(items)} 个数字向量 (维度={dimension})")

已向索引添加 10 个数字向量 (维度=1)


In [15]:
# 查询的数据
query_number = 205001020500
query_vec = [float(query_number)]  # 1 维查询向量

k = 5

results = client.query_points(
    collection_name=collection_name, 
    query = query_vec, 
    limit = k,
    with_payload  = True
    ).points
results

[ScoredPoint(id=9, version=1, score=0.0, payload={'type': 'order', 'id': '订单3002', 'number': 205001020777}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=8, version=1, score=0.0, payload={'type': 'order', 'id': '订单3001', 'number': 205001020333}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=5, version=1, score=1988870100.0, payload={'type': 'order', 'id': '订单2001', 'number': 203012150045}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4, version=1, score=1990000600.0, payload={'type': 'order', 'id': '订单1002', 'number': 203011010123}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=1990000600.0, payload={'type': 'order', 'id': '订单1001', 'number': 203011010001}, vector=None, shard_key=None, order_value=None)]

In [ ]:
results = client.query_points(
    collection_name=collection_name, 
    query = query_vec, 
    limit = k,
    query_filter=Filter(
        must=[FieldCondition(key="type", match=MatchValue(value="order"))]
    ),
    with_payload  = True
    ).points
results

[ScoredPoint(id=9, version=1, score=0.0, payload={'type': 'order', 'id': '订单3002', 'number': 205001020777}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=8, version=1, score=0.0, payload={'type': 'order', 'id': '订单3001', 'number': 205001020333}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=5, version=1, score=1988870100.0, payload={'type': 'order', 'id': '订单2001', 'number': 203012150045}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=1990000600.0, payload={'type': 'order', 'id': '订单1001', 'number': 203011010001}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4, version=1, score=1990000600.0, payload={'type': 'order', 'id': '订单1002', 'number': 203011010123}, vector=None, shard_key=None, order_value=None)]